# DocTamper 蒸馏评测（Colab）

使用 DTD reproduction 相同测试设置：`DocTamperV1-FCD` + `minq=75`。

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
%cd /content
if not os.path.exists('/content/DocTamper'):
    !git clone -b feat-distillation-fixpath-colab https://github.com/LeSiIence/DocTamper.git
%cd /content/DocTamper/models
!pip install -q lmdb albumentations segmentation_models_pytorch timm efficientnet_pytorch tqdm
!pip install -q opencv-python-headless Pillow
!git clone https://github.com/dwgoon/jpegio.git
%cd jpegio
!python setup.py install -q
%cd /content/DocTamper

In [ ]:
import os

# 把测试集放在 Drive 目录下，例如 /content/drive/MyDrive/TargetFolder/DocTamperV1-FCD
TARGET_FOLDER = '/content/drive/MyDrive/TargetFolder'
DATASET_NAME = 'DocTamperV1-FCD'
DATASET_SRC = f'{TARGET_FOLDER}/{DATASET_NAME}'
DATASET_DST = f'/content/DocTamper/{DATASET_NAME}'

if not os.path.exists(DATASET_DST):
    !cp -r "$DATASET_SRC" "$DATASET_DST"

assert os.path.exists(DATASET_DST), f'测试集不存在: {DATASET_DST}'
assert os.path.exists('/content/DocTamper/qt_table.pk'), '缺少 qt_table.pk'
assert os.path.exists('/content/DocTamper/pks'), '缺少 pks 目录'
os.makedirs('/content/DocTamper/pths', exist_ok=True)
print('测试集准备完成')

In [ ]:
import os

# 直接从 Drive 拷贝蒸馏权重到本地 pths
# 示例：/content/drive/MyDrive/TargetFolder/light_dtd_distill.pth
TARGET_FOLDER = '/content/drive/MyDrive/TargetFolder'
DISTILL_CKPT_NAME = 'light_dtd_distill.pth'
CKPT_SRC = f'{TARGET_FOLDER}/{DISTILL_CKPT_NAME}'
LOCAL_CKPT = '/content/DocTamper/pths/light_dtd_distill_eval.pth'

assert os.path.exists(CKPT_SRC), f'蒸馏权重不存在: {CKPT_SRC}'
os.makedirs('/content/DocTamper/pths', exist_ok=True)
!cp "$CKPT_SRC" "$LOCAL_CKPT"

assert os.path.exists(LOCAL_CKPT), f'权重拷贝失败: {LOCAL_CKPT}'
print('权重拷贝完成:', LOCAL_CKPT)

In [ ]:
%cd /content/DocTamper
!CUDA_VISIBLE_DEVICES=0 python -m models.eval_light_dtd --data_root /content/DocTamper --lmdb_name DocTamperV1-FCD --pth /content/DocTamper/pths/light_dtd_distill_eval.pth --minq 75 --batch_size 6 --num_workers 2